# Figure 3 — Field-dependent trihydrogen formationPanel a: time-of-flight spectra binned by surface electric field, pooled overall particle sizes and compositions.Panel b: relative yields of H+, H2+ and H3+ versus field, with exponential fitsand shot-level bootstrap errors.Panel c: representative CDI patterns classified as monomer and dimer.Panel d: monomer and dimer spectra at matched effective field.Outputs: `figures/data/fig3a_*.npy`, `fig3b_*.npy`, `fig3c_*.npy`, `fig3d_*.npy`.

> **Data.** These notebooks read the preprocessed per-run HDF5 files produced by> `scripts/preprocess.py`. Point `EUXFEL_NANO_DATA` at the directory holding> them. The raw data is at <https://doi.org/10.22003/XFEL.EU-DATA-003408-00>,> under embargo until publication.>> Figures 1-3 can be rebuilt **without** this data from the arrays already in> `figures/data/` — see `figures/README.md`.

In [ ]:
import sys
sys.path.insert(0, "../src")

import numpy as np
import pandas as pd
import xarray as xr
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib import gridspec, cm
from matplotlib.colors import LogNorm
from matplotlib.lines import Line2D
from scipy import constants
from scipy.optimize import curve_fit

from euxfel_nano import config
from euxfel_nano.constants import TIME_BETWEEN_PULSES, CHANNELS_PER_PULSE, channel_time
from euxfel_nano.io import read, read_ion, read_pnccd, data_root
from euxfel_nano.selection import (ion_selection, select_right_of_line,
                                   fish_plot_ion_selection)
from euxfel_nano.calibration import apply_calibration, mq_of, tof_of
from euxfel_nano.fields import build_charge_map, build_charge_maps
from euxfel_nano.yields import (FieldBinnedData, get_yields, significance,
                                verify_errors, sat_func, dec_func,
                                hist_for_selection)

P = config.params()
print("data root:", data_root())

## Selections and per-shot surface fieldThe field is computed per shot from the integrated eTOF photoelectron signal(SI Eq. 2), using the **TEM-measured** particle diameters — the nominal sizesare labels only. `build_charge_maps` defaults to the measured values in`config/runs.yaml`.

In [ ]:
# Ion-count window for the main analysis (SI: 500-2000).
NEVENTS = tuple(P["ion_count_windows"]["main"])
print("ion-count window:", NEVENTS)

DIAMETERS = config.measured_diameters_m()
for k, d in DIAMETERS.items():
    print(f"  {k:>4}: {d*1e9:.0f} nm")

In [ ]:
# Per-type event tables and eTOF traces, restricted to the ion-count window and
# to the region right of the dividing line.
divider = P["spatial"]["divider_line"]
THRESHOLD = [(500, 5000)]

event_dfs, bounded_dfevents, etofs = {}, {}, {}
for key, nc_name in [("500", "selected_etof500.nc"), ("300", "selected_etof300.nc"),
                     ("100", "selected_etof100.nc"), ("cs", "selected_etof_coreshell.nc")]:
    dfevent, dfpulse = ion_selection(config.run_ids(key), THRESHOLD)[0]
    etof = xr.open_dataarray(nc_name)
    dfevent, dfpulse, etof = select_right_of_line(dfevent, dfpulse, etof,
                                                  divider[0], divider[1])
    bounded_dfpulse = dfpulse[(dfpulse.nevents_pulse > NEVENTS[0])
                              & (dfpulse.nevents_pulse < NEVENTS[1])]
    bounded = dfevent[dfevent.trainId.isin(bounded_dfpulse.trainId)]
    event_dfs[key], bounded_dfevents[key], etofs[key] = dfevent, bounded, etof

charge_maps = {k: build_charge_map(etofs[k], bounded_dfevents[k], DIAMETERS[k])
               for k in event_dfs}
for k, m in charge_maps.items():
    print(f"{k:>4}: n={len(m):6d}  field {m.min():6.2f} - {m.max():7.2f} V/nm  "
          f"median {m.median():.2f}")

In [ ]:
data = FieldBinnedData(
    event_dfs=event_dfs,
    charge_maps=charge_maps,
    charge_windows=config.field_bins(),
    tof_range=tuple(P["analysis_tof_range_s"]),
)
print("field bin centres:", data.field_centers)

## Panel a — spectra binned by field

In [ ]:
tof_range = data.tof_range
bins_per_window = P["bins_per_field_window"]
colors = cm.viridis(np.linspace(0.05, 0.95, len(data.charge_windows)))

plt.figure(figsize=(12, 6))
for (low, high), NBINS, col in zip(data.charge_windows, bins_per_window, colors):
    bins = np.linspace(tof_range[0], tof_range[1], NBINS + 1)
    total = np.zeros(NBINS, dtype=float)
    for key in data.keys:
        cmap = charge_maps[key]
        sel = cmap.index[(cmap.values >= low) & (cmap.values < high)].values
        total += hist_for_selection(event_dfs[key], sel, bins, tof_range)
    if total.sum() == 0:
        continue
    y = total * NBINS / total.sum()
    x = 1e9 * 0.5 * (bins[:-1] + bins[1:])
    plt.plot(x, y, color=col, lw=2.0, label=f"{low:g}-{high:g} V/nm")
    tag = f"{low:g}-{high:g}"
    np.save(f"../figures/data/fig3a_x_{tag}.npy", x)
    np.save(f"../figures/data/fig3a_y_{tag}.npy", y)

plt.xlabel("Time of flight (ns)", fontsize=16)
plt.ylabel("Normalized counts (arb. unit)", fontsize=16)
plt.xlim(120, 490); plt.legend(fontsize=12); plt.show()

## Panel b — relative yields versus fieldErrors are 1-sigma bootstrap over shots: within each field bin the pulseidentifiers are resampled with replacement 4000 times and the pooled fractionrecomputed. The shot is the independent statistical unit, not the ion.

In [ ]:
integration_ranges = config.species_windows_ns()
styles = {"H+":   {"color": "#1f77b4", "marker": "o", "ls": "--"},
          "H2+":  {"color": "#d62728", "marker": "s", "ls": ":"},
          "H3+":  {"color": "#2ca02c", "marker": "^", "ls": "-."}}
# Filenames keep the LaTeX labels the figure scripts expect.
FILE_LABEL = {"H+": "H$^+$", "H2+": "H$_2^+$", "H3+": "H$_3^+$"}

E_smooth = np.linspace(data.field_centers.min(), data.field_centers.max(), 500)
plt.figure(figsize=(8, 6))

for species, span in integration_ranges.items():
    raw_yield, err_raw, err_low = get_yields(data, span)
    model = dec_func if species == "H2+" else sat_func
    popt, _ = curve_fit(model, data.field_centers, raw_yield,
                        sigma=np.clip(err_raw, 1e-6, None), absolute_sigma=True,
                        p0=[raw_yield.max() - raw_yield.min(), 1.5, raw_yield.min()],
                        maxfev=10000)
    ref = raw_yield[0]
    plt.errorbar(data.field_centers, raw_yield / ref, yerr=err_low,
                 fmt=styles[species]["marker"], color=styles[species]["color"],
                 markersize=8, capsize=3, elinewidth=1.2, zorder=3)
    plt.plot(E_smooth, model(E_smooth, *popt) / ref,
             color=styles[species]["color"], ls=styles[species]["ls"], lw=2.0)

    lab = FILE_LABEL[species]
    np.save(f"../figures/data/fig3b_raw_yield_{lab}.npy", raw_yield)
    np.save(f"../figures/data/fig3b_err_raw_{lab}.npy", err_raw)
    np.save(f"../figures/data/fig3b_err_low_{lab}.npy", err_low)

np.save("../figures/data/fig3b_field_centers.npy", data.field_centers)
plt.xlabel("Surface electric field (V/nm)", fontsize=16)
plt.ylabel("Normalized relative yield", fontsize=16)
plt.ylim(0.7, None); plt.show()

### Significance and error-model validationReproduces the numbers quoted in the SI: amplitudes differing from zero at28.9 sigma (H+), 10.2 (H2+) and 8.7 (H3+), with a likelihood-ratio comparisonagainst a constant giving p < 1e-3 for each.

In [ ]:
for species, span in integration_ranges.items():
    A, sA, nsig, dchi2, p = significance(data, span)
    print(f"{species:5s} amplitude {A:8.5f} +/- {sA:.5f}  ->  {nsig:5.1f} sigma"
          f"   dchi2={dchi2:8.1f}  p={p:.2e}")

In [ ]:
# Full four-way validation of the error model (SI checks i-iv, plus
# convergence and significance). Slow: 4000 resamples per bin per check.
for species, span in integration_ranges.items():
    print("\n#### " + species + " ####")
    verify_errors(data, span)

## Panels c and d — monomer / dimerMorphology labels come from the CNN of `notebooks/classifier.ipynb`. For dimersthe effective field adds the neighbour's contribution by superposition at theseparation measured from the two-particle interference fringes.

In [ ]:
trainIds_monomers300 = np.load('full_trainIds_monomers300.npy')
trainIds_dimers300 = np.load('full_trainIds_dimers300.npy')

mono_ion_dfevent300 = long_outside_spatial_selected_dfevent300[long_outside_spatial_selected_dfevent300.trainId.isin(trainIds_monomers300)]
mono_ion_dfpulse300 = long_outside_spatial_selected_dfpulse300[long_outside_spatial_selected_dfpulse300.trainId.isin(trainIds_monomers300)]

dimer_ion_dfevent300 = long_outside_spatial_selected_dfevent300[long_outside_spatial_selected_dfevent300.trainId.isin(trainIds_dimers300)]
dimer_ion_dfpulse300 = long_outside_spatial_selected_dfpulse300[long_outside_spatial_selected_dfpulse300.trainId.isin(trainIds_dimers300)]

In [ ]:
mono_etof300 = long_outside_spatial_selected_etof300.sel(pulseId=long_outside_spatial_selected_etof300.coords['pulseId'].isin(mono_ion_dfpulse300.pulseId))
dimer_etof300 = long_outside_spatial_selected_etof300.sel(pulseId=long_outside_spatial_selected_etof300.coords['pulseId'].isin(dimer_ion_dfpulse300.pulseId))

In [ ]:
x_lower, x_upper = (1e-7, 2e-7)
tof_lower, tof_upper = int(x_lower/channel_time), int(x_upper/channel_time)

def build_charge_map(selected_etof, bounded_dfevent, d):
    """Same recipe as charge500/300/... : surface Coulomb field in V/nm."""
    short = selected_etof.isel(data=slice(tof_lower, tof_upper))
    etof_sums = -np.sum(short, axis=1)
    H_counts = bounded_dfevent.pulseId.value_counts()
    ids = np.intersect1d(H_counts.index, etof_sums.pulseId.values)
    num_e = etof_sums.sel(pulseId=ids).values
    field = num_e*constants.e / (4*constants.pi*constants.epsilon_0*(d/2)**2) * 1e-9
    return pd.Series(field, index=ids)

# ---- FILL THESE IN ----------------------------------------
d_mono  = 300e-9   
d_dimer = 300e-9          
charge_map_mono  = build_charge_map(mono_etof300,  mono_ion_dfevent300,  d_mono)
charge_map_dimer = build_charge_map(dimer_etof300, dimer_ion_dfevent300, d_dimer)
# -----------------------------------------------------------

# CHECK 0: mapping produced finite, sensible fields
for name, m in [("mono", charge_map_mono), ("dimer", charge_map_dimer)]:
    print(f"{name}: n={len(m)}  field min={m.min():.2f} max={m.max():.2f} "
          f"median={m.median():.2f} V/nm  (NaNs: {m.isna().sum()})")

In [ ]:
# ============================================================
# STEP 1 — find the field band with most overlapping statistics
# ============================================================
band_edges = np.arange(0, 12.5, 0.5)
hm,_ = np.histogram(charge_map_mono.values,  bins=band_edges)
hd,_ = np.histogram(charge_map_dimer.values, bins=band_edges)
overlap = np.minimum(hm, hd)
best = np.argmax(overlap)
print("field band   mono / dimer / min-overlap")
for i in range(len(band_edges)-1):
    flag = "  <-- most overlap" if i==best else ""
    print(f"  {band_edges[i]:.1f}-{band_edges[i+1]:.1f}: {hm[i]:5d} / {hd[i]:5d} / {overlap[i]:5d}{flag}")

# ============================================================
# STEP 2 — select that band (set from STEP 1 output)
# ============================================================
field_band = (2.0, 4.0)   # <-- adjust to the most-overlap region above

def select_band(cmap, dfevent, band):
    ids = cmap.index[(cmap.values >= band[0]) & (cmap.values < band[1])].values
    ev = dfevent[dfevent.pulseId.isin(ids)]
    ev = ev[ev.tof.between(1e-7, 5e-7, inclusive='neither')]
    return ev, ids

mono_sel,  mono_ids  = select_band(charge_map_mono,  mono_ion_dfevent300,  field_band)
dimer_sel, dimer_ids = select_band(charge_map_dimer, dimer_ion_dfevent300, field_band)

# CHECK 1: enough events?
print(f"\nin {field_band} V/nm:  mono {len(mono_sel)} events / {len(mono_ids)} pulses,"
      f"  dimer {len(dimer_sel)} events / {len(dimer_ids)} pulses")
assert len(mono_sel)>100 and len(dimer_sel)>100, "too few events — widen field_band"

# CHECK 2: the whole point — fields now actually match
mf = charge_map_mono[(charge_map_mono>=field_band[0])&(charge_map_mono<field_band[1])]
df = charge_map_dimer[(charge_map_dimer>=field_band[0])&(charge_map_dimer<field_band[1])]
print(f"selected field median:  mono {mf.median():.2f}  dimer {df.median():.2f} V/nm"
      f"  (want these close)")

# ============================================================
# STEP 3 — shared-grid histogram, same normalization as your TOF plot
# ============================================================
xlimits = (1e-7, 5e-7); NBINS = 150
bins = np.linspace(*xlimits, NBINS+1)
x = 0.5*(bins[:-1]+bins[1:])*1e9
mh,_ = np.histogram(mono_sel.tof,  bins=bins)
dh,_ = np.histogram(dimer_sel.tof, bins=bins)
y_mono  = mh*NBINS/mh.sum();  std_mono  = np.sqrt(mh)*NBINS/mh.sum()
y_dimer = dh*NBINS/dh.sum();  std_dimer = np.sqrt(dh)*NBINS/dh.sum()

species_colors={'H':'#1f77b4','H2':'#d62728','H3':'#2ca02c'}
species_ranges={'H':(180.,225.),'H2':(265.,325.),'H3':(347.,393.)}
plt.figure(figsize=(12,6))
plt.plot(x, y_mono,  label='Monomer', lw=2.5, color='#5a4a3a')
plt.plot(x, y_dimer, label='Dimer',   lw=2.5, color='#c9a635')
for sp,(lo,hi) in species_ranges.items():
    plt.axvline(lo,color=species_colors[sp],ls='--',lw=1.5,alpha=0.6)
    plt.axvline(hi,color=species_colors[sp],ls='--',lw=1.5,alpha=0.6)
plt.xlabel('Time of flight (ns)', fontsize=18)
plt.ylabel('Normalized counts (arb. unit)', fontsize=18)
plt.xlim(120,490); plt.ylim(0,6.1)
plt.tick_params(axis='both', labelsize=14)
plt.legend(fontsize=22, loc='upper right'); plt.tight_layout(); plt.show()

In [ ]:
np.save("../figures/data/fig3d_x.npy", x)
np.save("../figures/data/fig3d_y_mono.npy", y_mono)
np.save("../figures/data/fig3d_y_dimer.npy", y_dimer)